In [3]:
#Imports

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier, LGBMRegressor

!pip install econml
from econml.dr import DRLearner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 13.5 MB/s eta 0:00:00
  Attempting uninstall: shap
    Found existing installation: shap 0.52.0
    Uninstalling shap-0.52.0:
      Successfully uninstalled shap-0.52.0


In [4]:
#Upload Dataset
from google.colab import files
import io

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))
  csv_file = io.BytesIO(uploaded[fn])
  df = pd.read_csv(csv_file)

print("Data loaded successfully! Here are the first 5 rows:")
display(df.head())

Saving Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv to Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv
User uploaded file "Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv" with length 4546990 bytes
Data loaded successfully! Here are the first 5 rows:


,id,event_type,latitude,longitude,endlatitude,endlongitude,address,end_address,event_cause,requires_road_closure,...,resolved_at_address,resolved_at_latitude,resolved_at_longitude,closed_by_id,closed_datetime,resolved_by_id,resolved_datetime,gba_identifier,zone,junction
0,FKID000000,unplanned,13.040004,77.518099,0.000000,0.000000,"Mumbai Bengaluru Highway, Jalahalli Cross Junc...",NaN,vehicle_breakdown,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,FKID000001,unplanned,12.921876,77.645158,0.000000,0.000000,"19th Main Road, Heavie Halcyon, Agara, HSR Lay...",NaN,vehicle_breakdown,False,...,"19th Main Road, Heavie Halcyon, Agara, HSR Lay...",12.921876,77.645158,NaN,NaN,FKUSR00002,2024-01-30 04:17:46.828355+00,NaN,NaN,NaN
2,FKID000002,unplanned,12.955622,77.585708,0.000000,0.000000,"Lalbagh Main Road, Dr Sri Shantaveera Swami Ci...",NaN,others,False,...,NaN,NaN,NaN,FKUSR00003,2024-01-30 04:56:03.281509+00,NaN,NaN,Bengaluru Central Corporation,Central Zone 2,UrvashiJunction
3,FKID000003,unplanned,13.006147,77.579435,13.006239,77.579516,"Sankey Road, Bashyam Circle, Sadashiva Nagar, ...","Sankey Road, Palace Orchard Upper, Sadashiva N...",tree_fall,True,...,NaN,NaN,NaN,FKUSR00004,2024-03-14 07:42:05.54944+00,NaN,NaN,NaN,NaN,NaN
4,FKID000004,unplanned,12.953980,77.585233,0.000000,0.000000,"Lalbagh Fort Road, Lalbagh Main Gate Junction,...",NaN,vehicle_breakdown,False,...,NaN,NaN,NaN,FKUSR00003,2024-01-30 05:35:17.338283+00,NaN,NaN,NaN,NaN,LalbaghMainGateJunc


In [7]:
#Create a Priority number
df["priority_num"] = df["priority"].map({"High": 2, "Low": 1}).fillna(1).astype(int)

#Booleanise road closure
df["closure_num"] = df["requires_road_closure"].astype(int)

#using priority number and closure to create severity score

# severity_score is an ordinal 0–3:
#   0 = Low priority, no closure   (minor event)
#   1 = High priority, no closure  (notable but flowing)
#   2 = Low priority, closure      (road blocked, lower-traffic corridor)
#   3 = High priority, closure     (full gridlock scenario)


df["severity_score"] = df["closure_num"] * 2 + (df["priority_num"] - 1)

print("\nSeverity score distribution:")
print(df["severity_score"].value_counts().sort_index())


Severity score distribution:
severity_score
0    2764
1    4733
2     379
3     297
Name: count, dtype: int64


In [8]:
 #Confounder feature engineering


df["start_dt"] = pd.to_datetime(df["start_datetime"], utc=True, errors="coerce")

# Temporal features
df["hour"]        = df["start_dt"].dt.hour.fillna(12).astype(int)
df["day_of_week"] = df["start_dt"].dt.dayofweek.fillna(0).astype(int)
df["month"]       = df["start_dt"].dt.month.fillna(1).astype(int)
df["is_weekend"]  = (df["day_of_week"] >= 5).astype(int)

# Peak hour: 7–9 AM and 5–8 PM Bengaluru rush hours
df["is_peak_hour"] = df["hour"].isin([7, 8, 9, 17, 18, 19, 20]).astype(int)

# Spatial / administrative encodings
label_encoders = {}
for col in ["corridor", "zone", "police_station"]:
    le = LabelEncoder()
    df[col + "_enc"] = le.fit_transform(df[col].fillna("Unknown"))
    label_encoders[col] = le

# Keep raw corridor name for output labelling
df["corridor_name"] = df["corridor"].fillna("Unknown")
df["zone_name"]     = df["zone"].fillna("Unknown")

# Geo coordinates (lat/lon give spatial signal for spillover)
df["latitude"]  = df["latitude"].fillna(df["latitude"].median())
df["longitude"] = df["longitude"].fillna(df["longitude"].median())

# Drop rows with missing start time (can't featurise)
df = df.dropna(subset=["hour"]).reset_index(drop=True)
print(f"\nRows after cleaning: {len(df)}")

FEATURE_COLS = [
    "hour", "day_of_week", "month", "is_weekend", "is_peak_hour",
    "corridor_enc", "zone_enc", "police_station_enc",
    "latitude", "longitude",
]

X = df[FEATURE_COLS].values
Y = df["severity_score"].values.astype(float)




Rows after cleaning: 8173


In [9]:
#treating

In [10]:
df["W"] = (df["event_type"] == "planned").astype(int)
W = df["W"].values

print(f"\nTreatment split — Planned (W=1): {W.sum()}, Unplanned (W=0): {(W==0).sum()}")
print(f"Naive severity gap: {Y[W==1].mean():.3f} (planned) vs {Y[W==0].mean():.3f} (unplanned)")



Treatment split — Planned (W=1): 467, Unplanned (W=0): 7706
Naive severity gap: 1.313 (planned) vs 0.749 (unplanned)


In [11]:
propensity_model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    num_leaves=15,
    random_state=42,
    verbose=-1,
)

outcome_model = LGBMRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    num_leaves=15,
    random_state=42,
    verbose=-1,
)

# DRLearner wraps both models and handles the DR correction internally
dr = DRLearner(
    model_propensity=propensity_model,
    model_regression=outcome_model,
    model_final=LGBMRegressor(n_estimators=100, max_depth=3, verbose=-1),
    cv=3,           # 3-fold cross-fitting (standard for DR)
    random_state=42,
)

print("\nFitting Doubly Robust estimator (this takes ~30–60 seconds)...")
dr.fit(Y, W, X=X)
print("Done.")



Fitting Doubly Robust estimator (this takes ~30–60 seconds)...
Done.


In [13]:
cate = dr.effect(X)  # shape: (n_samples,)

df["cate"] = cate
df["cate_lower"] = cate - 0.2
# CI via bootstrap is slow; using ±0.2 point-estimate band for demo
df["cate_upper"] = cate + 0.2

print(f"\nGlobal ATE (average causal effect of planned events): {cate.mean():.4f}")
print(f"This means planned events cause +{cate.mean():.2f} severity units on average,")
print(f"controlling for corridor, time, and location.")



Global ATE (average causal effect of planned events): 0.5230
This means planned events cause +0.52 severity units on average,
controlling for corridor, time, and location.


In [14]:
corridor_stats = df.groupby("corridor_name").agg(
    event_count       = ("id", "count"),
    mean_cate         = ("cate", "mean"),
    planned_rate      = ("W", "mean"),
    closure_rate      = ("closure_num", "mean"),
    mean_severity     = ("severity_score", "mean"),
    zone              = ("zone_name", lambda x: x.mode()[0]),
    lat               = ("latitude", "mean"),
    lon               = ("longitude", "mean"),
).reset_index()

# Normalise each component to 0–1 then weighted sum
def minmax(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

corridor_stats["risk_score"] = (
    0.4 * minmax(corridor_stats["mean_cate"])        # causal uplift weight (highest)
  + 0.3 * minmax(corridor_stats["closure_rate"])     # actual closure history
  + 0.2 * minmax(corridor_stats["mean_severity"])    # base severity
  + 0.1 * minmax(corridor_stats["planned_rate"])     # how often planned events hit it
)

corridor_stats["risk_rank"] = corridor_stats["risk_score"].rank(ascending=False).astype(int)
corridor_stats = corridor_stats.sort_values("risk_score", ascending=False)

corridor_stats.to_csv("corridor_risk_scores.csv", index=False)
print(f"\nSaved corridor_risk_scores.csv ({len(corridor_stats)} corridors)")

print("\n── TOP 15 CORRIDORS BY CAUSAL RISK SCORE ──")
print(corridor_stats[[
    "corridor_name", "zone", "mean_cate", "closure_rate",
    "planned_rate", "risk_score", "risk_rank"
]].head(15).to_string(index=False))



Saved corridor_risk_scores.csv (23 corridors)

── TOP 15 CORRIDORS BY CAUSAL RISK SCORE ──
         corridor_name         zone  mean_cate  closure_rate  planned_rate  risk_score  risk_rank
                 CBD 1      Unknown   0.844582      0.115385      0.038462    0.770866          1
Airport New South Road North Zone 1   0.446325      0.104478      0.253731    0.724414          2
           Mysore Road      Unknown   0.334843      0.110363      0.037685    0.681669          3
           Tumkur Road      Unknown   1.686481      0.026201      0.008734    0.628364          4
           ORR North 1      Unknown   0.469039      0.080000      0.065455    0.619696          5
      Old Airport Road      Unknown   0.404951      0.078947      0.118421    0.617892          6
            ORR East 1      Unknown   0.364852      0.073770      0.032787    0.580110          7
                 CBD 2      Unknown   0.448503      0.067308      0.057692    0.578582          8
 IRR(Thanisandra road)    

In [15]:


# Build X matrices for counterfactual prediction
# We use the outcome model directly for this
X_treat   = np.column_stack([X, np.ones(len(X))])   # W=1 world
X_control = np.column_stack([X, np.zeros(len(X))])  # W=0 world

# Predict Y under each world (using fitted propensity+outcome via effect)
# effect() already gives Y(1)-Y(0), but we also want absolute predictions
# Use the DR's internal outcome model for this:
from sklearn.ensemble import GradientBoostingRegressor

# Fit a simple outcome model directly for counterfactual absolutes
outcome_simple = LGBMRegressor(n_estimators=300, max_depth=5, verbose=-1)
X_with_W = np.column_stack([X, W])
outcome_simple.fit(X_with_W, Y)

X_as_planned   = np.column_stack([X, np.ones(len(X))])
X_as_unplanned = np.column_stack([X, np.zeros(len(X))])

df["pred_severity_if_planned"]   = outcome_simple.predict(X_as_planned)
df["pred_severity_if_unplanned"] = outcome_simple.predict(X_as_unplanned)
df["counterfactual_delta"] = df["pred_severity_if_planned"] - df["pred_severity_if_unplanned"]

# Export planned-event table
planned_df = df[df["event_type"] == "planned"][[
    "id", "start_datetime", "event_cause", "corridor_name", "zone_name",
    "priority", "requires_road_closure", "severity_score",
    "pred_severity_if_planned", "pred_severity_if_unplanned",
    "counterfactual_delta", "cate", "cate_lower", "cate_upper",
    "latitude", "longitude",
]].copy()

planned_df = planned_df.sort_values("counterfactual_delta", ascending=False)
planned_df.to_csv("counterfactual_table.csv", index=False)
print(f"\nSaved counterfactual_table.csv ({len(planned_df)} planned events)")

print("\nTop 10 highest-impact planned events:")
print(planned_df[["event_cause","corridor_name","zone_name",
                  "severity_score","counterfactual_delta"]].head(10).to_string())









def predict_impact(
    corridor: str,
    event_cause: str,
    hour: int,
    day_of_week: int,
    month: int,
    lat: float,
    lon: float,
    is_planned: bool = True,
) -> dict:
    """
    Predict causal impact of a future event.

    Returns:
        severity_if_happens   : predicted severity score (0–3)
        severity_if_rerouted  : counterfactual severity
        causal_delta          : extra severity caused by the event
        risk_level            : 'Critical' / 'High' / 'Medium' / 'Low'
        recommendation        : action string for the deployment manifest
    """
    corridor_enc = (
        label_encoders["corridor"].transform([corridor])[0]
        if corridor in label_encoders["corridor"].classes_
        else 0
    )

    # Build feature row (no zone/police_station available in hypothetical)
    x_row = np.array([[
        hour, day_of_week, month,
        int(day_of_week >= 5),                       # is_weekend
        int(hour in [7, 8, 9, 17, 18, 19, 20]),      # is_peak_hour
        corridor_enc,
        0,   # zone_enc (unknown)
        0,   # police_station_enc (unknown)
        lat, lon,
    ]])

    w_planned   = np.column_stack([x_row, [[1]]])
    w_unplanned = np.column_stack([x_row, [[0]]])

    sev_planned   = float(outcome_simple.predict(w_planned)[0])
    sev_unplanned = float(outcome_simple.predict(w_unplanned)[0])
    delta = sev_planned - sev_unplanned

    if delta >= 1.5:
        risk = "Critical"
        rec  = f"Pre-deploy 3+ officers to {corridor} at T-60 min. Request diversion route."
    elif delta >= 1.0:
        risk = "High"
        rec  = f"Pre-deploy 2 officers to {corridor} at T-45 min."
    elif delta >= 0.5:
        risk = "Medium"
        rec  = f"Alert nearest zone officer. Monitor {corridor}."
    else:
        risk = "Low"
        rec  = "Standard monitoring. No pre-deployment needed."

    return {
        "corridor":             corridor,
        "event_cause":          event_cause,
        "severity_if_happens":  round(sev_planned, 3),
        "severity_if_rerouted": round(sev_unplanned, 3),
        "causal_delta":         round(delta, 3),
        "risk_level":           risk,
        "recommendation":       rec,
    }





Saved counterfactual_table.csv (467 planned events)

Top 10 highest-impact planned events:
       event_cause corridor_name       zone_name  severity_score  counterfactual_delta
4771  public_event  Non-corridor     West Zone 1               2              1.621585
4707  public_event  Non-corridor     West Zone 1               2              1.621585
1676  vip_movement  Non-corridor         Unknown               2              1.616447
1979  vip_movement  Non-corridor         Unknown               2              1.608051
1641  vip_movement  Non-corridor         Unknown               2              1.606516
2017  vip_movement  Non-corridor         Unknown               2              1.606516
1896  vip_movement  Non-corridor         Unknown               2              1.587046
1934  vip_movement  Non-corridor         Unknown               2              1.535450
6648  construction  Non-corridor  Central Zone 2               2              1.534077
6647  construction  Non-corridor  Cent

In [16]:
 # ───Demo: predict impact of a hypothetical protest on Mysore Road at 6 PM ───
print("\n── WHAT-IF DEMO ──")
result = predict_impact(
    corridor="Mysore Road",
    event_cause="protest",
    hour=18,
    day_of_week=4,   # Friday
    month=3,
    lat=12.97,
    lon=77.57,
    is_planned=True,
)
for k, v in result.items():
    print(f"  {k:28s}: {v}")

result2 = predict_impact(
    corridor="ORR East 1",
    event_cause="procession",
    hour=9,
    day_of_week=0,   # Monday
    month=4,
    lat=12.92,
    lon=77.64,
    is_planned=True,
)
print()
for k, v in result2.items():
    print(f"  {k:28s}: {v}")

print("\n✓ Stage 1 complete. Outputs: corridor_risk_scores.csv + counterfactual_table.csv")
print("  → Feed corridor_risk_scores.csv into Stage 2 (MARL) as the risk score state vector.")


── WHAT-IF DEMO ──
  corridor                    : Mysore Road
  event_cause                 : protest
  severity_if_happens         : 1.925
  severity_if_rerouted        : 1.829
  causal_delta                : 0.096
  risk_level                  : Low
  recommendation              : Standard monitoring. No pre-deployment needed.

  corridor                    : ORR East 1
  event_cause                 : procession
  severity_if_happens         : 1.804
  severity_if_rerouted        : 1.047
  causal_delta                : 0.758
  risk_level                  : Medium
  recommendation              : Alert nearest zone officer. Monitor ORR East 1.

✓ Stage 1 complete. Outputs: corridor_risk_scores.csv + counterfactual_table.csv
  → Feed corridor_risk_scores.csv into Stage 2 (MARL) as the risk score state vector.
